## Import

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"  

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.cuda.get_device_name(0), torch.cuda.mem_get_info())

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[0]   
sys.path.insert(0, str(ROOT / "src"))

from ResonanSC.bulk import get_batch_names, get_bulk, get_multimodal_bulk, get_subtype_onehot, prune_empty_cols
from ResonanSC.model import LearnPseudoMaker
from ResonanSC.corr import plot_corr_heatmaps
from ResonanSC.mapping import mapping_init_from_weights, prepare_multimodal_training_data
from ResonanSC.merge_align import (
    run_inbatch_merge,
    run_cross_batch_align,
    build_M_align
)

## Preprocess

In [ ]:
import scanpy as sc
import scanpy.external as sce
# adata = sc.read_h5ad("data/processed/result1/init_adata.h5ad")  
adata = sc.read_h5ad("data/processed/init_adata.h5ad")
adata

In [ ]:
batch_key = 'batch'
type_key = 'major_subset'

In [ ]:
batch_list = adata.obs[batch_key].unique()
genedata={}
for i, b in enumerate(batch_list):
    ad_i = adata[adata.obs[batch_key] == b].copy()
    genedata[i] = ad_i
    print(i, b, ad_i)

## Train

In [ ]:
batch_names = get_batch_names(genedata, col=batch_key)
batch_names

In [ ]:
from pathlib import Path
import torch

from ResonanSC.mapping import mapping_init_from_weights

# train_path = Path("outputs/result1/1/train_mapping_1.pt") 
init_path = Path("outputs/result3/init.pt")
# train_path = Path("outputs/result2/1(old)/train_M_3.pt")
# init_path = Path("outputs/result2/1(old)/init.pt")


train_ckpt = torch.load(train_path, map_location="cpu", weights_only=False)
init_ckpt = torch.load(init_path, map_location="cpu", weights_only=False)

# 补齐训练 checkpoint 缺少的上下文
train_ckpt["P_align"] = train_ckpt["probs_soft"]
train_ckpt["M_align"] = train_ckpt["M"]
train_ckpt["align_masks"] = init_ckpt["align_masks"]

train_ckpt["mapping_init"] = mapping_init_from_weights(
    train_ckpt.get("mapping_init", init_ckpt["mapping_init"]),
    train_ckpt["mapping_weights"],
)

for key in [
    "gene_names",
    "batch_key",
    "batch_names",
    "training_batch_order",
    "training_data_dir",
]:
    if key in init_ckpt:
        train_ckpt.setdefault(key, init_ckpt[key])

train_ckpt["mapping_max_distance"] = init_ckpt.get(
    "dap_deg_window",
    init_ckpt.get("mapping_max_distance", 250_000),
)

fixed_path = train_path.with_name(train_path.stem + "_complete.pt")
torch.save(train_ckpt, fixed_path)

print("saved:", fixed_path)
print("keys:", sorted(train_ckpt.keys()))

In [ ]:
P_init = None
M_init = None

# init_path = "outputs/result1/1/merge_6.5.pt"   # init.pt or merge_*.pt
init_path = "outputs/result3/init.pt"  #outputs/test/init.pt"
# init_path = "outputs/test/init.pt" #"outputs/result2/1(old)/init.pt"
pckpt = torch.load(init_path, map_location="cpu", weights_only=False)
# Use globally aligned probabilities for training. P_merge can have a
# different local label count per batch after in-batch merges, while
# M_align/align_masks are in the global label space.
P_init = pckpt.get("P_align", pckpt.get("P_init"))
M_init = pckpt.get("M_prob_align", pckpt.get("M_prob", pckpt.get("M_align", pckpt.get("M", None))))
align_masks = pckpt["align_masks"]
mapping_init = pckpt["mapping_init"]
mapping_max_distance = pckpt.get("dap_deg_window", 250_000)
gene_names = pckpt["gene_names"]
batch_key = pckpt.get("batch_key", batch_key)
print("mapping_max_distance:", mapping_max_distance)

if P_init is None:
    raise ValueError("Checkpoint must contain P_align or P_init.")

expected_k = int(M_init.shape[1]) if M_init is not None else int(align_masks["bulknum"])
bad_shapes = [tuple(p.shape) for p in P_init if p.shape[1] != expected_k]
if bad_shapes:
    raise ValueError(
        "P_init must be in the same global label space as M_init/align_masks; "
        f"expected {expected_k} columns, got {bad_shapes}. "
        "Use P_align/P_init instead of P_merge for training."
    )

# Preferred path: 01_Initialize saves the final filtered training dict directly.
training_data_dir = Path(pckpt.get(
    "training_data_dir",
    "data/processed/02_intermediate/training_multidata_preinit"
    # "data/processed/result1/training_multidata",   #"data/processed/training_multidata"
))

def h5ad_index_key(path):
    return int(path.name.split("_", 1)[0])

training_files = sorted(training_data_dir.glob("*.h5ad"), key=h5ad_index_key)
if training_files:
    multidata = {i: sc.read_h5ad(path) for i, path in enumerate(training_files)}
else:
    # Backward compatibility with older checkpoints: restore DA-peak matrices
    # and replace ATAC gene-activity batches.
    atac_da_dir = Path(pckpt.get(
        "atac_da_dir",
        "data/processed/result1/atacdata_da",
    ))
    atac_files = sorted(atac_da_dir.glob("*.h5ad"), key=h5ad_index_key)
    if not atac_files:
        raise FileNotFoundError(f"No training h5ad files found in {training_data_dir} or {atac_da_dir}")
    atacdata_da = {i: sc.read_h5ad(path) for i, path in enumerate(atac_files)}
    multidata = prepare_multimodal_training_data(
        initialized_data=genedata,
        atacdata_da=atacdata_da,
        batch_key=batch_key,
    )

batch_names = get_batch_names(multidata, col=batch_key)
for i, ad_i in multidata.items():
    modality = "ATAC peak" if "atac" in str(ad_i.obs[batch_key].iloc[0]).lower() else "RNA gene"
    print(i, batch_names[i], modality, ad_i.shape)


In [ ]:
margin = {
    "learn_M": [0.3, 1.5, 3.0],
    "learn_mapping": [0.3, 1.5, 3.0],
    "learn_P": [0.5, 2.0, 10.0],
}

round = 1
output_dir = Path("outputs/result3")
output_dir.mkdir(parents=True, exist_ok=True)
path_P = output_dir / f"train_P_{round}.pt"
path_mapping = output_dir / f"train_mapping_{round}.pt"
path_M = output_dir / f"train_M_{round}.pt"

common_args = dict(
    rnadata=multidata,
    P_init=P_init,
    M_init=M_init,
    align_masks=align_masks,
    batch_keys=batch_names,
    batch_key=batch_key,
    gene_names=gene_names,
    mapping_init=mapping_init,
    mapping_max_distance=mapping_max_distance,
    one_hot_start=False,
    present_eps=0.1,
    rna_chunk_size=512,
    cellbulk_chunk_size=512,
    cache_rna_on_device=False,
)

# if round == 1:
#    resume_for_P = None
# else:
#     resume_for_P = path


### train P

In [ ]:
# Step 1: optimize cell-type assignments P.
out_P = LearnPseudoMaker(
    **common_args,
    epochs=1,
    lr=1e-2,
    margin=margin["learn_P"],
    lambdas=[10.0, 10.0, 1.0],
    stage="learn_P",
    verbose_every=10,
    resume_path= path_M #output_dir / f"train_M_{round-1}.pt"
)
# torch.save(out_P, path_P)

### train mapping

In [ ]:
# Step 2: optimize batch-specific nonnegative peak-gene mappings.
out_mapping = LearnPseudoMaker(
    **common_args,
    epochs=100,
    lr=1e-3,
    margin=margin["learn_mapping"],
    lambdas=[1.0, 1.0, 0.0],
    mapping_lambdas=[1.0, 1.0],  # distance, cross-batch mapping consistency
    stage="learn_mapping",
    verbose_every=10,
    resume_path=path_M,
)
torch.save(out_mapping, path_mapping)

### train M

In [ ]:
# Step 3: optimize cell-type-specific binary marker matrix M.
out_M = LearnPseudoMaker(
    **common_args,
    epochs=100,
    lr=1e-2,
    margin=margin["learn_M"],
    lambdas=[1.0, 1.0, 0.0],
    marker_sparsity=1.0,
    stage="learn_M",
    verbose_every=10,
    resume_path="outputs/result2/1(old)/train_M_1.pt",
)
torch.save(out_M, path_M)

# To continue alternating optimization, set resume_for_P = path_M and rerun
# the three cells in P -> mapping -> M order.

### get outputs

In [ ]:
# Directly restore this round's training output from a saved checkpoint.
# This avoids constructing the model or running another epoch, which can OOM.
checkpoint_path = Path("outputs/result3/train_M_1.pt")
# If path_M is already set to the checkpoint you want, this is equivalent:
# checkpoint_path = path_M

train_out = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
print("loaded checkpoint:", checkpoint_path)
print("training_stage:", train_out.get("training_stage"))
print("history epochs:", len(train_out.get("history", [])))

probs_soft = train_out["probs_soft"]
M_hard = train_out["M"]
M_prob = train_out.get("M_prob", M_hard)
# Use continuous M for diagnostics/merge/next-round init so learn_M progress
# is not lost before probabilities cross the hard 0.5 threshold.
M_for_merge = M_prob
M = M_hard
bulks_list = train_out["bulks_list"]
history = train_out["history"]

## Results

In [ ]:
# import scanpy as sc
# import numpy as np
# from ResonanSC.merge_align import write_p_labels_by_obs_names

# write_p_labels_by_obs_names(
#     multidata,
#     probs_soft,
#     adata,
#     key="pred",
#     batch_key=batch_key,
# )

sc.pl.umap(adata, color=['minor_subset','pred'],wspace=0.5)
sc.pl.umap(adata, color=[batch_key])

In [ ]:
adata.write("data/processed/01_intermediate/result.h5ad")

In [ ]:
adata.obs['pred'].value_counts()

In [ ]:
import pandas as pd

out = adata.obs[["pred"]].copy()
out.index.name = "cell_barcode"
out.to_csv(output_dir/ "pred_with_barcodes.csv")

In [ ]:
corr_heatmap = plot_corr_heatmaps(
    mode='weighted',
    bulks_list=bulks_list,
    M_list=[M_for_merge for _ in bulks_list],
    batch_names=batch_names,
    figsize_per_panel=(9,9),
    diag_use_mean=True,
)

In [ ]:
p_hard = []
for p in probs_soft:
    hard = torch.argmax(p, dim=1)
    ph = torch.zeros_like(p)
    ph[torch.arange(p.shape[0]), hard] = 1.0
    p_hard.append(ph)

bulks_hard = get_multimodal_bulk(
    multidata,
    p_hard,
    train_out=train_out,
    batch_key=batch_key,
)

corr_hard = plot_corr_heatmaps(
    mode="weighted",
    bulks_list=bulks_hard,
    M_list=[M_for_merge for _ in bulks_hard],
    batch_names=batch_names,
    figsize_per_panel=(9, 9),
    diag_use_mean=True,
)

### marker diagnolise

In [ ]:
M_hard = train_out["M"]
M_prob = train_out.get("M_prob")

print("hard marker counts:", (M_hard > 0).sum(0).tolist())

if M_prob is not None:
    print("prob > 0.5:", (M_prob > 0.5).sum(0).tolist())
    print("prob > 0.8:", (M_prob > 0.8).sum(0).tolist())
    print("prob min/mean/max:", M_prob.min().item(), M_prob.mean().item(), M_prob.max().item())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 10, figsize=(20, 9))
axes = axes.flatten()

for i in range(M.shape[1]):
    col = M_prob[:, i]                         # (2000,)
    print(len(col[col>0.5]))

    axes[i].hist(col.detach().cpu().numpy(), bins=50)
    axes[i].set_title(f"Column {i} >0 markers")
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import scanpy as sc
from scipy import sparse


if hasattr(M, "detach"):
    M_np = M.detach().cpu().numpy()
else:
    M_np = np.asarray(M)


# 2) 取 X 并算 score
X = adata.copy().X
if sparse.issparse(X):
    X_score = X @ M_np                 
else:
    X_score = np.asarray(X) @ M_np

# 3) 存到 obsm
adata.obsm["X_mscore"] = X_score

print("X_mscore shape:", adata.obsm["X_mscore"].shape)  # (n_cells, bulknum)

# sc.pp.pca(adata, n_comps=20)
# sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca")
# sc.tl.umap(adata)

bulknum = adata.obsm["X_mscore"].shape[1]
score_names = []
for ct in range(bulknum):
    cname = f"mscore_{ct}"
    adata.obs[cname] = adata.obsm["X_mscore"][:, ct]
    score_names.append(cname)

adata.obsm['X_umap'] = adata.obsm['X_umap'].copy()

sc.pl.umap(
    adata,
    color=[batch_key, type_key, "pred"],
    wspace=0.4
)

X_m = adata.obsm["X_mscore"]  # (n_cells, bulknum)

vmin = np.percentile(X_m, 1)
vmax = np.percentile(X_m, 99)

sc.pl.umap(
    adata,
    color=score_names,
    ncols=5,
    vmin=vmin,
    vmax=vmax,
    cmap="viridis",
    wspace=0.4
)

### mapping diagnolise

In [ ]:
from ResonanSC.mapping_plot import mapping_edge_table

mapping_df = mapping_edge_table(train_out)
mapping_df.head()

In [ ]:
import matplotlib.pyplot as plt

from ResonanSC.mapping_plot import (
    mapping_gene_totals,
    plot_gene_mapping_tracks,
    plot_mapping_diagnostics,
)

# General diagnostics: distance-weight scatter, cross-batch comparison,
# gene-by-batch heatmap and cell-type-specific effective activity.
mapping_diagnostics = plot_mapping_diagnostics(
    train_out=train_out,
    multidata=multidata,
    batch_key=batch_key,
    top_n_genes=40,
)

if mapping_diagnostics["batch_scatter"] is not None:
    mapping_batch_stats = mapping_diagnostics["batch_scatter"][3]
    display(mapping_batch_stats)

# Show peak-level effective contributions for the gene receiving the
# largest average normalized mapping weight. Replace this with any gene.
gene_totals = mapping_gene_totals(train_out)
gene_to_plot = gene_totals.mean(axis=1).idxmax()
gene_track = plot_gene_mapping_tracks(
    train_out=train_out,
    multidata=multidata,
    batch_key=batch_key,
    gene=gene_to_plot,
    top_n=30,
)
print("gene track:", gene_to_plot)
plt.show()

In [ ]:
import pandas as pd
EPS = 1e-8

def _numpy(value):
    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    return np.asarray(value)


def save_train_mapping(
    train_out,
    output_dir,
    prefix: str = "mapping",
):
    """Save learned peak-gene mappings as tables and a lossless torch file.

    The compressed edge table is intended for inspection with pandas. The
    torch file preserves the original sparse mapping tensors for later reuse.
    """
    required = {"mapping_init", "mapping_weights"}
    missing = sorted(required.difference(train_out))
    if missing:
        raise KeyError(f"train_out is missing required keys: {missing}")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    table_input = dict(train_out)
    table_input["mapping_init"] = train_out.get(
        "mapping_init_before_training",
        train_out["mapping_init"],
    )
    edges = mapping_edge_table(table_input)
    edges["weight_delta"] = edges["weight"] - edges["init_weight"]
    edges["weight_fold"] = edges["weight"] / (edges["init_weight"] + EPS)
    edges["gene_rank_for_peak"] = (
        edges.groupby(["batch_id", "peak_idx"])["weight"]
        .rank(method="first", ascending=False)
        .astype(np.int64)
    )
    edges = edges.sort_values(
        ["batch_id", "peak_idx", "gene_rank_for_peak"],
        ignore_index=True,
    )

    gene_totals = mapping_gene_totals(train_out, normalize=False).reset_index()
    batch_summary = (
        edges.groupby(["batch_id", "batch"], as_index=False)
        .agg(
            n_edges=("weight", "size"),
            n_peaks=("peak_idx", "nunique"),
            n_genes=("gene_idx", "nunique"),
            init_weight_mean=("init_weight", "mean"),
            learned_weight_mean=("weight", "mean"),
            learned_weight_median=("weight", "median"),
            learned_weight_sum=("weight", "sum"),
        )
    )

    paths = {
        "edges": output_dir / f"{prefix}_edges.csv.gz",
        "gene_totals": output_dir / f"{prefix}_gene_totals.csv",
        "batch_summary": output_dir / f"{prefix}_batch_summary.csv",
        "torch": output_dir / f"{prefix}.pt",
    }
    edges.to_csv(paths["edges"], index=False, compression="gzip")
    gene_totals.to_csv(paths["gene_totals"], index=False)
    batch_summary.to_csv(paths["batch_summary"], index=False)

    torch.save(
        {
            "mapping_init": train_out["mapping_init"],
            "mapping_init_before_training": train_out.get(
                "mapping_init_before_training"
            ),
            "mapping_weights": train_out["mapping_weights"],
            "gene_names": train_out.get("gene_names"),
            "batch_names": train_out.get("batch_names"),
            "source_train_checkpoint": train_out.get("source_train_checkpoint"),
        },
        paths["torch"],
    )

    print(f"[mapping] saved {len(edges):,} edges to {paths['edges']}")
    return {
        "paths": paths,
        "edges": edges,
        "gene_totals": gene_totals,
        "batch_summary": batch_summary,
    }

def mapping_gene_totals(train_out, normalize: bool = True) -> pd.DataFrame:
    """Aggregate mapping weights to globally normalized gene totals per batch."""
    gene_names = list(map(str, train_out["gene_names"]))
    result = pd.DataFrame(index=pd.Index(gene_names, name="gene"))

    for batch_id in sorted(train_out["mapping_weights"]):
        info = train_out["mapping_init"][batch_id]
        gene_idx = _numpy(info["gene_idx"]).astype(np.int64)
        weights = _numpy(train_out["mapping_weights"][batch_id]).astype(np.float64)
        totals = np.bincount(gene_idx, weights=weights, minlength=len(gene_names))
        if normalize:
            totals = totals / (totals.sum() + EPS)
        result[str(info.get("batch_name", batch_id))] = totals

    return result

def mapping_edge_table(train_out) -> pd.DataFrame:
    """Return one row per trainable peak-gene edge."""
    rows = []
    mapping_init = train_out["mapping_init"]
    mapping_weights = train_out["mapping_weights"]

    for batch_id in sorted(mapping_weights):
        info = mapping_init[batch_id]
        peak_idx = _numpy(info["peak_idx"]).astype(np.int64)
        gene_idx = _numpy(info["gene_idx"]).astype(np.int64)
        distance = _numpy(info["distance"]).astype(np.float64)
        weight = _numpy(mapping_weights[batch_id]).astype(np.float64)
        init_weight = _numpy(info["init_weight"]).astype(np.float64)
        peak_names = np.asarray(info["peak_names"], dtype=object)
        gene_names = np.asarray(info["gene_names"], dtype=object)

        rows.append(
            pd.DataFrame(
                {
                    "batch_id": int(batch_id),
                    "batch": str(info.get("batch_name", batch_id)),
                    "peak_idx": peak_idx,
                    "gene_idx": gene_idx,
                    "peak": peak_names[peak_idx],
                    "gene": gene_names[gene_idx],
                    "distance": distance,
                    "init_weight": init_weight,
                    "weight": weight,
                }
            )
        )

    if not rows:
        raise ValueError("train_out does not contain any ATAC mapping weights.")
    return pd.concat(rows, ignore_index=True)


In [ ]:
from pathlib import Path
# from ResonanSC.mapping_plot import save_train_mapping

mapping_result = save_train_mapping(
    train_out,
    output_dir=Path("outputs/result3/mapping"),
    prefix="round_1",
)

display(mapping_result["gene_totals"])
display(mapping_result["edges"].head())

## fine

In [ ]:
probs_soft = p_align
bulks_list = bulks_list_align
key_pred = 'pred_align'

In [ ]:
import math
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


def detect_outliers_mad(x, thresh=3.5):
    x = np.asarray(x).ravel()
    if len(x) == 0:
        return np.array([], dtype=bool), np.array([], dtype=float)

    med = np.median(x)
    mad = np.median(np.abs(x - med))

    if mad < 1e-12:
        return np.zeros(len(x), dtype=bool), np.zeros(len(x), dtype=float)

    robust_z = 0.6745 * (x - med) / mad
    is_outlier = np.abs(robust_z) > thresh
    return is_outlier, robust_z


def write_obs_columns_by_obs_names(data_dict, adata_all, cols, batch_key=None):
    """类似 write_p_labels_by_obs_names：按 obs_names / (batch, obs_name) 写回 adata_all.obs。"""
    pieces = []

    for batch, ad in data_dict.items():
        missing = [c for c in cols if c not in ad.obs]
        if missing:
            raise KeyError(f"Batch {batch} missing obs columns: {missing}")

        piece = pd.DataFrame({"_obs_name": np.asarray(ad.obs_names)})
        for col in cols:
            piece[col] = np.asarray(ad.obs[col])

        if batch_key is not None:
            if batch_key not in ad.obs:
                raise KeyError(f"Batch {batch} has no obs column {batch_key!r}.")
            piece["_batch"] = np.asarray(ad.obs[batch_key])

        pieces.append(piece)

    frame = pd.concat(pieces, ignore_index=True)

    if batch_key is None and "batch" in adata_all.obs and all(
        "batch" in ad.obs for ad in data_dict.values()
    ):
        batch_key = "batch"
        frame["_batch"] = np.concatenate(
            [np.asarray(ad.obs[batch_key]) for ad in data_dict.values()]
        )

    if batch_key is None:
        source_index = frame["_obs_name"]
        target_index = adata_all.obs_names
    else:
        if batch_key not in adata_all.obs:
            raise KeyError(f"adata_all has no obs column {batch_key!r}.")
        source_index = pd.MultiIndex.from_arrays(
            [frame["_batch"], frame["_obs_name"]],
            names=["_batch", "_obs_name"],
        )
        target_index = pd.MultiIndex.from_arrays(
            [np.asarray(adata_all.obs[batch_key]), np.asarray(adata_all.obs_names)],
            names=["_batch", "_obs_name"],
        )

    if source_index.has_duplicates:
        dup = source_index[source_index.duplicated()].unique()[:10].tolist()
        raise ValueError(f"Duplicated keys when writing obs columns: {dup}")

    values = frame.set_index(source_index)[cols]
    aligned = values.reindex(target_index)

    if aligned.isna().any().any():
        missing = aligned.index[aligned.isna().any(axis=1)][:10].tolist()
        raise ValueError(
            f"Some cells in adata are missing outlier labels; examples: {missing}"
        )

    for col in cols:
        adata_all.obs[col] = aligned[col].to_numpy()

    return aligned


# =========================
# 0. 基本检查
# =========================
keys = list(multidata.keys())

if len(keys) != len(probs_soft):
    raise ValueError(f"multidata长度({len(keys)})和probs_soft长度({len(probs_soft)})不一致")

if len(batch_names) != len(probs_soft):
    raise ValueError(f"batch_names长度({len(batch_names)})和probs_soft长度({len(probs_soft)})不一致")


# =========================
# 1. 先得到硬分类
# =========================
hard = [
    torch.argmax(p, dim=1).detach().cpu().numpy()
    for p in probs_soft
]


# =========================
# 2. 画图准备
# =========================
n_plots = len(probs_soft)
n_cols = math.ceil(np.sqrt(n_plots))
n_rows = math.ceil(n_plots / n_cols)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(5 * n_cols, 4 * n_rows),
    squeeze=False,
)
axes = axes.flatten()

outlier_results = {}
legend_handles = {}
rng = np.random.default_rng(0)


# =========================
# 3. 主循环：基于 probs_soft 做 PCA/outlier
# =========================
for i, k in enumerate(keys):
    ax = axes[i]
    ad = multidata[k]

    p_np = probs_soft[i].detach().cpu().numpy()
    labels = hard[i]

    if ad.n_obs != p_np.shape[0]:
        raise ValueError(
            f"{k}: multidata[{k}].n_obs({ad.n_obs}) 和 probs_soft[{i}]行数({p_np.shape[0]})不一致，"
            f"说明细胞数量或顺序不对应"
        )

    unique_classes = np.unique(labels)
    n_classes = len(unique_classes)
    cmap = plt.cm.get_cmap("tab20", n_classes)

    is_outlier_all = np.zeros(ad.n_obs, dtype=bool)
    score_all = np.full(ad.n_obs, np.nan)
    pc1_all = np.full(ad.n_obs, np.nan)

    outlier_results[batch_names[i]] = {}

    for idx, cls in enumerate(unique_classes):
        mask = labels == cls
        cls_idx_global = np.where(mask)[0]
        X_cls = p_np[mask]

        if X_cls.shape[0] >= 2 and X_cls.shape[1] >= 1:
            x_1d = PCA(n_components=1).fit_transform(X_cls).flatten()
        else:
            x_1d = np.zeros(X_cls.shape[0])

        is_outlier, robust_z = detect_outliers_mad(x_1d, thresh=3.5)
        outlier_idx_global = cls_idx_global[is_outlier]

        outlier_results[batch_names[i]][int(cls)] = outlier_idx_global

        is_outlier_all[cls_idx_global] = is_outlier
        score_all[cls_idx_global] = robust_z
        pc1_all[cls_idx_global] = x_1d

        y = np.full(len(x_1d), idx, dtype=float) + rng.normal(0, 0.04, len(x_1d))

        pts = ax.scatter(
            x_1d[~is_outlier],
            y[~is_outlier],
            s=18,
            alpha=0.75,
            color=cmap(idx),
            label=f"class {cls}",
            edgecolors="none",
        )

        if np.any(is_outlier):
            ax.scatter(
                x_1d[is_outlier],
                y[is_outlier],
                s=18,
                alpha=0.75,
                color=cmap(idx),
                edgecolors="black",
                linewidths=0.8,
            )

        if cls not in legend_handles:
            legend_handles[cls] = pts

    # 先写回每个 batch 自己的 AnnData，供后续 reassign_outliers_1nn 使用
    ad.obs["hard_class"] = labels.astype(str)
    ad.obs["is_outlier"] = is_outlier_all
    ad.obs["outlier_score"] = score_all
    ad.obs["outlier_pc1"] = pc1_all

    ax.set_title(f"{batch_names[i]}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("Class")
    ax.set_yticks(range(n_classes))
    ax.set_yticklabels([str(c) for c in unique_classes])


# =========================
# 4. 按 obs_names 写回原始 adata，避免 concat 顺序问题
# =========================
write_obs_columns_by_obs_names(
    multidata,
    adata,
    cols=["hard_class", "is_outlier", "outlier_score", "outlier_pc1"],
    batch_key=batch_key,
)


# =========================
# 5. 删除多余子图 + 图例
# =========================
for j in range(n_plots, len(axes)):
    fig.delaxes(axes[j])

if len(legend_handles) > 0:
    sorted_classes = sorted(legend_handles.keys())
    handles = [legend_handles[c] for c in sorted_classes]
    labels_legend = [f"class {c}" for c in sorted_classes]

    fig.legend(
        handles,
        labels_legend,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        borderaxespad=0.0,
        frameon=False,
    )

plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

print("done")

In [ ]:
batch_id = 0
rnadata[keys[batch_id]].obs.head()
rnadata[keys[batch_id]].obs.groupby("hard_class")["is_outlier"].sum()

In [ ]:
key_pred = 'pred'

In [ ]:
import scanpy as sc

sc.pl.umap(adata, color=[type_key, key_pred], wspace=0.5)
sc.pl.umap(adata, color=[batch_key, "is_outlier"], wspace=0.5)

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neighbors import KNeighborsClassifier


def reassign_outliers_1nn(
    rnadata,
    bulks_list,
    key_pred,
    train_out,
    adata_all=None,
    proto_labels_list=None,
    metric="euclidean",
    batch_key="batch",
    rna_layer="norm",
):
    mapping_init = train_out["mapping_init"]
    mapping_weights = train_out["mapping_weights"]
    eps = 1e-8

    for i, ((batch, adata_batch), bulk) in enumerate(
        zip(rnadata.items(), bulks_list)
    ):
        mask = adata_batch.obs["is_outlier"].astype(bool).to_numpy()

        adata_batch.obs["pred_fine"] = adata_batch.obs[key_pred].astype(object)
        adata_batch.obs["outlier_fine"] = pd.Series(
            pd.NA, index=adata_batch.obs_names, dtype="object"
        )

        if not mask.any():
            continue

        batch_name = str(adata_batch.obs[batch_key].iloc[0]).lower()
        is_atac = "atac" in batch_name

        if is_atac:
            X = sparse.csr_matrix(adata_batch.X, dtype=np.float32)

            row_sum = np.asarray(X.sum(axis=1)).ravel()
            tf = X.multiply(1.0 / (row_sum[:, None] + eps)).tocsr()

            feature_sum = np.asarray(tf.sum(axis=0)).ravel()
            idf = np.log1p(X.shape[0] / (feature_sum + eps))
            tfidf = tf.multiply(idf).tocsr()

            info = mapping_init[i]
            weights = mapping_weights[i]

            peak_idx = np.asarray(
                info["peak_idx"].detach().cpu(), dtype=np.int64
            )
            gene_idx = np.asarray(
                info["gene_idx"].detach().cpu(), dtype=np.int64
            )
            weights = np.asarray(
                weights.detach().cpu(), dtype=np.float32
            )

            n_genes = len(info["gene_names"])
            mapping = sparse.coo_matrix(
                (weights, (peak_idx, gene_idx)),
                shape=(adata_batch.n_vars, n_genes),
            ).tocsr()

            X_gene = (tfidf @ mapping).tocsr()
            X_gene.data = np.log1p(X_gene.data)
            X_test = X_gene[mask].toarray()

        else:
            X = (
                adata_batch.layers[rna_layer]
                if rna_layer in adata_batch.layers
                else adata_batch.X
            )
            X_test = X[mask]
            X_test = (
                X_test.toarray()
                if sparse.issparse(X_test)
                else np.asarray(X_test)
            )

        bulk_np = (
            bulk.detach().cpu().numpy()
            if hasattr(bulk, "detach")
            else np.asarray(bulk)
        )

        if bulk_np.shape[0] == X_test.shape[1]:
            X_train = bulk_np.T
        elif bulk_np.shape[1] == X_test.shape[1]:
            X_train = bulk_np
        else:
            raise ValueError(
                f"Batch {batch}: cell features={X_test.shape[1]}, "
                f"bulk shape={bulk_np.shape}"
            )

        if proto_labels_list is None:
            y_train = np.arange(X_train.shape[0]).astype(str)
        else:
            y_train = np.asarray(proto_labels_list[batch]).astype(str)

        clf = KNeighborsClassifier(n_neighbors=1, metric=metric)
        clf.fit(X_train, y_train)
        pred_outlier = clf.predict(X_test)

        adata_batch.obs.loc[mask, "pred_fine"] = pred_outlier
        adata_batch.obs.loc[mask, "outlier_fine"] = pred_outlier

    # 按 obs_names / (batch, obs_name) 写回原始 adata，避免重新 concat 后顺序错乱
    if adata_all is not None:
        write_obs_columns_by_obs_names(
            rnadata,
            adata_all,
            cols=["pred_fine", "outlier_fine"],
            batch_key=batch_key,
        )

    return rnadata

In [ ]:
def write_obs_columns_by_obs_names(data_dict, adata_all, cols, batch_key=None):
    """按 obs_names / (batch, obs_name) 写回 adata_all.obs，允许列值本身为 NA。"""
    pieces = []

    for batch, ad in data_dict.items():
        missing = [c for c in cols if c not in ad.obs]
        if missing:
            raise KeyError(f"Batch {batch} missing obs columns: {missing}")

        piece = pd.DataFrame({"_obs_name": np.asarray(ad.obs_names)})
        for col in cols:
            piece[col] = np.asarray(ad.obs[col], dtype=object)

        piece["_present"] = True

        if batch_key is not None:
            if batch_key not in ad.obs:
                raise KeyError(f"Batch {batch} has no obs column {batch_key!r}.")
            piece["_batch"] = np.asarray(ad.obs[batch_key])

        pieces.append(piece)

    frame = pd.concat(pieces, ignore_index=True)

    if batch_key is None and "batch" in adata_all.obs and all(
        "batch" in ad.obs for ad in data_dict.values()
    ):
        batch_key = "batch"
        frame["_batch"] = np.concatenate(
            [np.asarray(ad.obs[batch_key]) for ad in data_dict.values()]
        )

    if batch_key is None:
        source_index = pd.Index(frame["_obs_name"], name="_obs_name")
        target_index = pd.Index(adata_all.obs_names, name="_obs_name")
    else:
        if batch_key not in adata_all.obs:
            raise KeyError(f"adata_all has no obs column {batch_key!r}.")

        source_index = pd.MultiIndex.from_arrays(
            [frame["_batch"], frame["_obs_name"]],
            names=["_batch", "_obs_name"],
        )
        target_index = pd.MultiIndex.from_arrays(
            [np.asarray(adata_all.obs[batch_key]), np.asarray(adata_all.obs_names)],
            names=["_batch", "_obs_name"],
        )

    if source_index.has_duplicates:
        dup = source_index[source_index.duplicated()].unique()[:10].tolist()
        raise ValueError(f"Duplicated keys when writing obs columns: {dup}")

    values = frame.set_index(source_index)[cols + ["_present"]]
    aligned = values.reindex(target_index)

    # 只检查“有没有匹配到这一行”，不检查业务列里的 NA
    missing_rows = aligned["_present"].isna()
    if missing_rows.any():
        examples = aligned.index[missing_rows][:10].tolist()
        raise ValueError(
            f"{missing_rows.sum()} cells in adata are missing rows from data_dict; "
            f"examples: {examples}"
        )

    for col in cols:
        adata_all.obs[col] = aligned[col].to_numpy()

    return aligned[cols]

In [ ]:
multidata = reassign_outliers_1nn(
    multidata,
    bulks_list,
    key_pred=key_pred,
    train_out=train_out,
    adata_all=adata,
    batch_key=batch_key,
)

In [ ]:
sc.pl.umap(adata, color=["pred_fine", "outlier_fine"], wspace=0.3)

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.sparse import issparse

def detect_outliers_mad(x, thresh=3.5):
    x = np.asarray(x).ravel()
    if len(x) == 0:
        return np.array([], dtype=bool), np.array([], dtype=float)

    med = np.median(x)
    mad = np.median(np.abs(x - med))

    if mad < 1e-12:
        return np.zeros(len(x), dtype=bool), np.zeros(len(x), dtype=float)

    robust_z = 0.6745 * (x - med) / mad
    is_outlier = np.abs(robust_z) > thresh
    return is_outlier, robust_z


# =========================
# 0. 基本检查与顺序约定
# =========================
keys = list(rnadata.keys())

if len(keys) != len(probs_soft):
    raise ValueError(f"rnadata长度({len(keys)})和probs_soft长度({len(probs_soft)})不一致")

if len(batch_names) != len(probs_soft):
    raise ValueError(f"batch_names长度({len(batch_names)})和probs_soft长度({len(probs_soft)})不一致")


# =========================
# 1. 从 probs_soft 得到硬分类
# =========================
hard = []
for p in probs_soft:
    hard.append(torch.argmax(p, dim=1).detach().cpu().numpy())


# =========================
# 2. 画图准备
# =========================
n_plots = len(probs_soft)
n_cols = math.ceil(np.sqrt(n_plots))
n_rows = math.ceil(n_plots / n_cols)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(5 * n_cols, 4 * n_rows),
    squeeze=False
)
axes = axes.flatten()

# 保存每个 batch、每个类的 outlier 行号
outlier_results = {}

legend_handles = {}
rng = np.random.default_rng(0)  # 固定抖动随机种子，方便复现


# =========================
# 3. 主循环：同一次循环里既写 obs 又画图
# =========================
for i, k in enumerate(keys):
    ax = axes[i]
    ad = rnadata[k]
    labels = hard[i]

    # 用 rnadata[k].X 做每类 PCA / outlier
    X = ad.X
    if issparse(X):
        X = X.toarray()
    else:
        X = np.asarray(X)

    # 关键检查：保证细胞顺序一一对应
    if X.shape[0] != len(labels):
        raise ValueError(
            f"{k}: ad.X行数({X.shape[0]})和hard[{i}]长度({len(labels)})不一致，"
            f"说明 rnadata 与 probs_soft 的细胞顺序或数量不对应"
        )

    unique_classes = np.unique(labels)
    n_classes = len(unique_classes)
    cmap = plt.cm.get_cmap("tab20", n_classes)

    # 预分配写回 obs 的列
    is_outlier_all = np.zeros(ad.n_obs, dtype=bool)
    score_all = np.full(ad.n_obs, np.nan)
    pc1_all = np.full(ad.n_obs, np.nan)

    outlier_results[batch_names[i]] = {}

    for idx, cls in enumerate(unique_classes):
        mask = (labels == cls)
        cls_idx_global = np.where(mask)[0]
        X_cls = X[mask]

        # 每类单独做1维PCA
        if X_cls.shape[0] >= 2 and X_cls.shape[1] >= 1:
            x_1d = PCA(n_components=1).fit_transform(X_cls).flatten()
        else:
            x_1d = np.zeros(X_cls.shape[0])

        # 每类内部用 MAD 找 outlier
        is_outlier, robust_z = detect_outliers_mad(x_1d, thresh=3.5)
        outlier_idx_global = cls_idx_global[is_outlier]

        # 保存结果
        outlier_results[batch_names[i]][int(cls)] = outlier_idx_global

        # 写入当前 batch 的 obs 对应位置
        is_outlier_all[cls_idx_global] = is_outlier
        score_all[cls_idx_global] = robust_z
        pc1_all[cls_idx_global] = x_1d

        # 作图：y轴按类别放置，加少量抖动
        y = np.full(len(x_1d), idx, dtype=float) + rng.normal(0, 0.04, len(x_1d))

        # 非 outlier
        pts = ax.scatter(
            x_1d[~is_outlier],
            y[~is_outlier],
            s=18,
            alpha=0.75,
            color=cmap(idx),
            label=f"class {cls}",
            edgecolors='none'
        )

        # outlier：原点加黑色描边
        if np.any(is_outlier):
            ax.scatter(
                x_1d[is_outlier],
                y[is_outlier],
                s=18,
                alpha=0.75,
                color=cmap(idx),
                edgecolors='black',
                linewidths=0.8
            )

        # 总图例只收一次
        if cls not in legend_handles:
            legend_handles[cls] = pts

    # 写回 obs
    ad.obs["hard_class"] = labels.astype(str)
    ad.obs["is_outlier"] = is_outlier_all
    ad.obs["outlier_score"] = score_all
    ad.obs["outlier_pc1"] = pc1_all

    # 子图设置
    ax.set_title(f"{batch_names[i]}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("Class")
    ax.set_yticks(range(n_classes))
    ax.set_yticklabels([str(c) for c in unique_classes])


# =========================
# 4. 删除多余空白子图
# =========================
for j in range(n_plots, len(axes)):
    fig.delaxes(axes[j])


# =========================
# 5. 总图例放右侧外部
# =========================
if len(legend_handles) > 0:
    sorted_classes = sorted(legend_handles.keys())
    handles = [legend_handles[c] for c in sorted_classes]
    labels_legend = [f"class {c}" for c in sorted_classes]

    fig.legend(
        handles,
        labels_legend,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        borderaxespad=0.0,
        frameon=False
    )

plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

print("done")

## benchmark

In [ ]:
import scanpy as sc
import torch

for i in range(len(rnadata)):
    hard = torch.argmax(probs_soft[i], dim=1).detach().cpu().numpy()
    rnadata[i].obs["pred"] = hard.astype(str)

    # 复制一份用于调整
    rnadata[i].obs["pred_tune"] = rnadata[i].obs["pred"].copy()

    # 定义 merge 规则
    merge_map = {
        "14":"1",
        "20":"1",
        "2":"1",
        "11":"9",
        "8":"6",
        "19":"6",
        "7":"6",
        "13":"10"
    }

    rnadata[i].obs["pred_tune"] = rnadata[i].obs["pred_tune"].replace(merge_map)

adata_all = sc.concat(rnadata.values(), axis=0)

sc.pl.umap(adata_all, color=[type_key, "pred", "pred_tune"], wspace=0.5)
sc.pl.umap(adata_all, color=[batch_key])

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

def compute_ari_nmi(adata, col1, col2, average_method="arithmetic"):
    x = adata.obs[col1]
    y = adata.obs[col2]

    mask = x.notna() & y.notna()
    x = x[mask].astype(str)
    y = y[mask].astype(str)

    ari = adjusted_rand_score(x, y)
    nmi = normalized_mutual_info_score(x, y, average_method=average_method)

    return {"ARI": ari, "NMI": nmi, "n_cells_used": mask.sum()}

In [ ]:
adata_sub = 'CD8.TEM','NK.CD16hi','CD8.NAIVE','cMono','CD4.TCM','CD4.NAIVE'

In [ ]:
res = compute_ari_nmi(adata, type_key, 'pred', average_method="arithmetic")
print(res)

In [ ]:
res = compute_ari_nmi(adata, type_key, 'pred_pre', average_method="arithmetic")
print(res)
res = compute_ari_nmi(adata, type_key, 'pred_merge', average_method="arithmetic")
print(res)

## merge and align

### small label merge

In [ ]:
import numpy as np
import scanpy as sc

# adata 是 init_adata.h5ad，也就是 RNA + ATAC gene activity 的共同 gene space
if "X_pca" not in adata.obsm:
    n_comps = min(50, adata.n_obs - 1, adata.n_vars - 1)
    sc.pp.pca(adata, n_comps=n_comps, random_state=0)

# rep_data 用来算 small pred cluster 的 centroid 距离
# 顺序严格跟 multidata / probs_soft / p_merge 对齐
rep_data = {}

for i in range(len(multidata)):
    batch_name = str(multidata[i].obs[batch_key].iloc[0])
    ad_rep = adata[adata.obs[batch_key].astype(str) == batch_name].copy()

    # 尽量按 multidata 的 obs_names 对齐
    if multidata[i].obs_names.isin(ad_rep.obs_names).all():
        ad_rep = ad_rep[multidata[i].obs_names].copy()
    else:
        if ad_rep.n_obs != multidata[i].n_obs:
            raise ValueError(
                f"Batch {i} ({batch_name}) rep_data n_obs={ad_rep.n_obs}, "
                f"multidata n_obs={multidata[i].n_obs}; cannot align."
            )

    rep_data[i] = ad_rep
    print(i, batch_name, "rep:", rep_data[i].shape, "train:", multidata[i].shape)

import torch

def merge_small_pred_clusters_by_rep(
    rep_data,
    p_list,
    M_list=None,
    min_cells=20,
    rep_key="X_pca",
    device="cpu",
    eps=1e-8,
):
    p_out = []
    M_out = [] if M_list is not None else None
    infos = []

    for i in range(len(p_list)):
        if rep_key not in rep_data[i].obsm:
            raise KeyError(f"rep_data[{i}].obsm missing {rep_key!r}")

        p = p_list[i].to(device)
        hard = torch.argmax(p, dim=1).detach().cpu().numpy()
        K = p.shape[1]

        if rep_data[i].n_obs != p.shape[0]:
            raise ValueError(
                f"Batch {i}: rep_data n_obs={rep_data[i].n_obs}, "
                f"p rows={p.shape[0]}"
            )

        counts = np.bincount(hard, minlength=K)
        small = set(np.where(counts < min_cells)[0].tolist())
        large = [k for k in range(K) if k not in small and counts[k] > 0]

        merge_to = {k: k for k in range(K)}

        if small and large:
            emb = np.asarray(rep_data[i].obsm[rep_key], dtype=np.float32)
            centroids = np.zeros((K, emb.shape[1]), dtype=np.float32)

            for k in range(K):
                mask = hard == k
                if mask.any():
                    centroids[k] = emb[mask].mean(axis=0)

            large_mat = centroids[large]
            large_norm = large_mat / (np.linalg.norm(large_mat, axis=1, keepdims=True) + eps)

            for k in sorted(small):
                if counts[k] == 0:
                    continue
                x = centroids[k] / (np.linalg.norm(centroids[k]) + eps)
                nearest = large[int(np.argmax(large_norm @ x))]
                merge_to[k] = nearest

        elif small:
            print(f"[small pred merge] batch {i}: all clusters have < {min_cells} cells; keep original.")

        groups_by_target = {}
        for old_k, target_k in merge_to.items():
            groups_by_target.setdefault(target_k, []).append(old_k)

        groups = [groups_by_target[k] for k in sorted(groups_by_target)]

        p_new = torch.zeros(p.shape[0], len(groups), device=device, dtype=p.dtype)
        for new_k, group in enumerate(groups):
            p_new[:, group and new_k] = p[:, group].sum(dim=1)
        p_out.append(p_new)

        if M_list is not None:
            M = M_list[i].to(device)
            m_new = torch.zeros(M.shape[0], len(groups), device=device, dtype=M.dtype)
            mass = p.sum(dim=0).clamp_min(eps)

            for new_k, group in enumerate(groups):
                w = mass[group]
                m_new[:, new_k] = (M[:, group] * w[None, :]).sum(dim=1) / w.sum()

            M_out.append(m_new)

        infos.append({
            "batch": i,
            "counts_before": counts.tolist(),
            "small_clusters": sorted([int(k) for k in small]),
            "merge_map": {int(k): int(v) for k, v in merge_to.items() if int(k) != int(v)},
            "groups": [[int(x) for x in g] for g in groups],
            "n_before": int(K),
            "n_after": int(len(groups)),
        })

    return p_out, M_out, infos

In [ ]:
def merge_small_pred_hard_by_rep(
    rep_data,
    p_list,
    M_list=None,
    min_cells=20,
    rep_key="X_pca",
    device="cpu",
    eps=1e-8,
):
    p_out = []
    M_out = [] if M_list is not None else None
    infos = []

    shared_M = M_list if torch.is_tensor(M_list) else None

    for i in range(len(p_list)):
        p = p_list[i].to(device)
        hard = torch.argmax(p, dim=1).detach().cpu().numpy()
        K = p.shape[1]

        emb = np.asarray(rep_data[i].obsm[rep_key], dtype=np.float32)
        counts = np.bincount(hard, minlength=K)

        small = set(np.where((counts < min_cells) & (counts > 0))[0].tolist())
        large = [k for k in range(K) if counts[k] >= min_cells]

        merge_to = {k: k for k in range(K)}

        if small and large:
            centroids = np.zeros((K, emb.shape[1]), dtype=np.float32)
            for k in range(K):
                mask = hard == k
                if mask.any():
                    centroids[k] = emb[mask].mean(axis=0)

            large_mat = centroids[large]
            large_norm = large_mat / (np.linalg.norm(large_mat, axis=1, keepdims=True) + eps)

            for k in sorted(small):
                x = centroids[k] / (np.linalg.norm(centroids[k]) + eps)
                nearest = large[int(np.argmax(large_norm @ x))]
                merge_to[k] = nearest

        hard_merged_old = np.array([merge_to[int(k)] for k in hard])

        kept_old_labels = sorted(np.unique(hard_merged_old).tolist())
        old_to_new = {old: new for new, old in enumerate(kept_old_labels)}
        hard_merged = np.array([old_to_new[int(k)] for k in hard_merged_old])

        p_new = torch.zeros(p.shape[0], len(kept_old_labels), device=device, dtype=p.dtype)
        p_new[torch.arange(p.shape[0], device=device), torch.as_tensor(hard_merged, device=device)] = 1.0
        p_out.append(p_new)

        if M_list is not None:
            M = shared_M.to(device) if shared_M is not None else M_list[i].to(device)
            m_new = torch.zeros(M.shape[0], len(kept_old_labels), device=device, dtype=M.dtype)

            for old, new in old_to_new.items():
                members = [k for k, v in merge_to.items() if v == old and counts[k] > 0]
                if len(members) == 0:
                    members = [old]
                w = torch.as_tensor(counts[members], device=device, dtype=M.dtype).clamp_min(eps)
                m_new[:, new] = (M[:, members] * w[None, :]).sum(dim=1) / w.sum()

            M_out.append(m_new)

        infos.append({
            "batch": i,
            "n_before": int(K),
            "n_after": int(len(kept_old_labels)),
            "small_clusters": sorted([int(k) for k in small]),
            "merge_map": {int(k): int(v) for k, v in merge_to.items() if int(k) != int(v)},
            "counts_before": counts.tolist(),
            "kept_old_labels": [int(x) for x in kept_old_labels],
        })

    return p_out, M_out, infos

In [ ]:
import scanpy as sc

if "X_pca" not in adata.obsm:
    sc.pp.pca(adata, n_comps=50, random_state=0)

rep_data = {}

for i, b in enumerate(batch_names):
    ad_rep = adata[adata.obs[batch_key].astype(str) == str(b)].copy()

    if ad_rep.n_obs != multidata[i].n_obs:
        raise ValueError(
            f"batch {i} {b}: adata has {ad_rep.n_obs} cells, "
            f"multidata has {multidata[i].n_obs} cells"
        )

    rep_data[i] = ad_rep
    print(i, b, rep_data[i].shape, multidata[i].shape)

In [ ]:
adata[adata.obs['batch']=='rna_1'].obs['pred'].value_counts()

In [ ]:
MIN_CELL = 92

In [ ]:
p_pre, M_pre, small_pred_merge_info = merge_small_pred_hard_by_rep(
    rep_data=rep_data,
    p_list=probs_soft,
    M_list=M_for_merge,
    min_cells=MIN_CELL,
    rep_key="X_pca",
    device="cpu",
)

for info in small_pred_merge_info:
    print(
        info["batch"],
        "n_before =", info["n_before"],
        "n_after =", info["n_after"],
        "small =", info["small_clusters"],
        "merge_map =", info["merge_map"],
    )

write_p_labels_by_obs_names(
    multidata,
    p_pre,
    adata,
    key="pred_pre",
    batch_key=batch_key,
)

res = compute_ari_nmi(adata, type_key, "pred_pre", average_method="arithmetic")
print(res)

sc.pl.umap(adata, color=["pred", "pred_pre"], wspace=0.3)
sc.pl.umap(adata, color=[batch_key, type_key], wspace=0.3)

### mannual

In [ ]:
from ResonanSC.merge_align import (
    manual_inbatch_merge,
    manual_cross_batch_align,
    write_p_labels_by_obs_names,
)
from ResonanSC.mapping import mapping_init_from_weights

In [ ]:
# 查看每个batch当前实际使用的label
for i, batch_name in enumerate(batch_names):
    active = torch.argmax(probs_soft[i], dim=1).unique().cpu().tolist()
    print(batch_name, "active labels:", sorted(active))

In [ ]:
sc.pl.umap(adata[adata.obs['batch']=='rna_2'], color=[type_key,'pred'],wspace=0.5)

In [ ]:
align_maps = {
    batch_name: {
        local_label: f"{batch_name}_{local_label}"
        for local_label in merged_labels[batch_name]
    }
    for batch_name in batch_names
}

### function

In [ ]:
in_corrs = corr_merge['inbatch_corrs']
cross_corrs = corr_merge['crossbatch_corrs']

In [ ]:
thresholds = [0.97,0.97,0.97,0.99,0.97,0.99,0.97,0.97,0.97,0.99,0.97,0.90]
out_merge = run_inbatch_merge(inbatch_corrs = in_corrs, p_list=p_pre, thresholds=thresholds, M=M_pre, device="cpu")
p_merge = out_merge['p_merge']
M_merge = out_merge['M_merge']

# (Optional) store merged hard labels for visualization
for i in range(len(multidata)):
    hard = torch.argmax(p_merge[i], dim=1).detach().cpu().numpy()
    multidata[i].obs['pred_merge'] = hard.astype(str)

write_p_labels_by_obs_names(
    multidata,
    p_merge,
    adata,
    key="pred_merge",
    batch_key=batch_key,
)


res = compute_ari_nmi(adata, type_key, 'pred_merge', average_method="arithmetic")
print(res)
sc.pl.umap(adata, color=["pred","pred_pre", "pred_merge"],wspace=0.3)
sc.pl.umap(adata, color=[batch_key, type_key],wspace=0.3)

In [ ]:
bulks_list_merge = get_multimodal_bulk(
    multidata,
    p_pre,
    train_out=train_out,
    batch_key=batch_key,
)

In [ ]:
corr_merge = plot_corr_heatmaps(
    mode='weighted',
    bulks_list=bulks_list_merge,
    M_list=M_pre,
    batch_names=batch_names,
    figsize_per_panel=(9,9),
    diag_use_mean=True,
)

In [ ]:
# DE-mode correlation is not valid for mixed RNA/ATAC here because
# ATAC batches in multidata are peak matrices, while M and bulks are
# in gene space after learned peak-gene mapping. Use weighted bulk
# correlation for merge/align diagnostics.
corr_merge_de = None

In [ ]:
in_corr_merge = corr_merge['inbatch_corrs']
cross_corr_merge = corr_merge['crossbatch_corrs']

In [ ]:
out = run_cross_batch_align(
    p_merge = p_merge,
    batch_names = batch_names,
    cross_corrs = cross_corr_merge,
    cap=3,
    null_score=-0.9,
    device="cpu",
)

p_align = out['p_align']
align_masks = out['align_masks']

write_p_labels_by_obs_names(
    multidata,
    p_align,
    adata,
    key="pred_align",
    batch_key=batch_key,
)

# data_all = sc.concat(list(multidata.values()), axis=0)

res = compute_ari_nmi(adata, type_key, 'pred_align', average_method="arithmetic")
print(res)

sc.pl.umap(adata, color=["pred", "pred_align"],wspace=0.5)
sc.pl.umap(adata, color=[batch_key, type_key],wspace=0)

In [ ]:
K = 13

p_align = []
for p in p_merge:
    C = p.shape[0]
    p_new = torch.zeros(C,K)
    for i in range(9):
        p_new[:,i] = p[:,i].clone()
    p_new[:,9] = p[:,9:].sum(axis=1).clone()
    print(p[:,9:])
    p_align.append(p_new)

p_align

### no align

In [ ]:
import importlib
from pathlib import Path
import ResonanSC.merge_align as merge_align

# 当前 kernel 已加载过旧模块时需要 reload
importlib.reload(merge_align)

merge_only_path = Path(
    "outputs/result2/1(old)/merge_3.5.pt"
)

merge_ckpt = merge_align.save_merge_only_checkpoint(
    output_path=merge_only_path,
    p_merge=p_pre,
    M_merge=M_pre,
    train_out=train_out,
    batch_names=batch_names,
    source_checkpoint=pckpt,
    batch_key=batch_key,
    training_data_dir=training_data_dir,
    mapping_max_distance=mapping_max_distance,
)

print("saved:", merge_only_path)
print("merged K per batch:", merge_ckpt["K_list"])
print("global K:", merge_ckpt["M_align"].shape[1])
print("checkpoint keys:", sorted(merge_ckpt))

### align

In [ ]:
bulks_list_align = get_multimodal_bulk(
    multidata,
    p_align,
    train_out=train_out,
    batch_key=batch_key,
)

In [ ]:
M_align = build_M_align(
    M_list=M_merge,
    label_map=out["label_map"],
    offsets=out["offsets"],
    device=device,
    eps=1e-8,
    normalize=False,  # 关键
)

print(M_align.shape)   # [G, K_new]

In [ ]:
corr_align = plot_corr_heatmaps(
    mode='weighted',
    bulks_list=bulks_list_align,
    M_list=[M_align for _ in bulks_list_align],
    batch_names=batch_names,
    figsize_per_panel=(9,9),
    diag_use_mean=True,
)

In [ ]:
mapping_init_align = mapping_init_from_weights(
    train_out["mapping_init"],
    train_out["mapping_weights"],
)

torch.save(
    {
        "P_merge": p_merge,
        "P_align": p_align,
        "M_align": M_align,
        "align_masks": align_masks,
        "mapping_init": mapping_init_align,
        "gene_names": gene_names,
        "batch_key": batch_key,
    },
    "outputs/result2/1(old)/merge_3.5.pt"
)
